# Sistema de recomendação NETFLIX

- O dataset utilizado tem origem no [Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows?resource=download) e contém informações de produções disponíveis na plataforma de streamming Netflix.

- O objetivo é identificar as 5 produções mais similares à uma de interesse, com o propósito de fazer recomendações das mesmas.

Este notebook foi escrito com a intenção de se fazer uma análise dos dados a serem utilizados.


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r"..\data\raw\netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


- Há 8807 linhas e 12 colunas no dataframe (que podemos ver como um conjunto de vetores, onde cada coluna é um vetor)

- Com exceção de "release_year", que é composta por inteiros, as outras colunas são todas compostas por strings. Note que "date_added" poderia ser formatada como data, mas a seguir ela será descartada, então não há a necessidade de preocupação.

Como o objetivo é entender as semelhanças entre os filmes e/ou séries, é preciso decidir quais características seriam importantes para uma comparação que faça sentido. Dessa forma, serão mantidas as colunas "show_id" (chave única dos dados), "title" (informação que traz o nome da produção), "director" (ter um mesmo diretor em diferentes produções faz com que ambas tenham qualidades próximas), "listed_in" (informações referentes a gêneros) e "description" (traz um resumo do que se trata a obra).

In [13]:
df = df[["show_id", "title", "director", "listed_in", "description"]]

In [15]:
# Verificando nulos
df.isnull().sum().sort_values(ascending=False)

director       2634
show_id           0
title             0
listed_in         0
description       0
dtype: int64

Como há um número significativo de diretores não definidos, para manter os dados, será feita um preenchimento dos nulos por uma string vazia.

In [16]:
df["director"] = df["director"].fillna("")
df.isnull().sum()

show_id        0
title          0
director       0
listed_in      0
description    0
dtype: int64

In [17]:
# Checando se há valores duplicados
df.duplicated().sum()

np.int64(0)

In [20]:
df["director"].unique()

<StringArray>
[              'Kirsten Johnson',                              '',
               'Julien Leclercq',                 'Mike Flanagan',
 'Robert Cullen, José Luis Ucha',                  'Haile Gerima',
               'Andy Devonshire',                'Theodore Melfi',
             'Kongkiat Komesiri',           'Christian Schwochow',
 ...
       'Saratswadee Wongsomphet',             'Kirati Nakintanon',
                   'Mark Risley',                   'James Brown',
                    'Ivona Juka',                        'Mu Chu',
       'Chandra Prakash Dwivedi',               'Majid Al Ansari',
                  'Peter Hewitt',                   'Mozez Singh']
Length: 4529, dtype: str

Existem casos com registros de mais de um diretor por produção. Para trabalhar com esses dados, eles serão dispostos em listas.

In [21]:
# Separamos a string da coluna description por espaço em branco
df['description'] = df['description'].apply(lambda x:x.split())

In [22]:
# Colocando as informações em listas

df['director'] = df['director'].apply(lambda x:x.split(","))
df['listed_in'] = df['listed_in'].apply(lambda x:x.split(","))

In [23]:
df.head()

,show_id,title,director,listed_in,description
0,s1,Dick Johnson Is Dead,[Kirsten Johnson],[Documentaries],"[As, her, father, nears, the, end, of, his, li..."
1,s2,Blood & Water,[],"[International TV Shows, TV Dramas, TV Myste...","[After, crossing, paths, at, a, party,, a, Cap..."
2,s3,Ganglands,[Julien Leclercq],"[Crime TV Shows, International TV Shows, TV ...","[To, protect, his, family, from, a, powerful, ..."
3,s4,Jailbirds New Orleans,[],"[Docuseries, Reality TV]","[Feuds,, flirtations, and, toilet, talk, go, d..."
4,s5,Kota Factory,[],"[International TV Shows, Romantic TV Shows, ...","[In, a, city, of, coaching, centers, known, to..."


Agora, pensando que será feita uma vetorização dos textos, os mesmos serão agrupados caso sejam compostos. 

Por exemplo: "Nome Sobrenome" transforma-se em "NomeSobrenome"

In [24]:
# Replaces de espaço por vazio (remove o espaço)
df['director'] = df['director'].apply(lambda x:[i.replace(" ","") for i in x])
df['listed_in'] = df['listed_in'].apply(lambda x:[i.replace(" ","") for i in x])

In [25]:
df.head()

,show_id,title,director,listed_in,description
0,s1,Dick Johnson Is Dead,[KirstenJohnson],[Documentaries],"[As, her, father, nears, the, end, of, his, li..."
1,s2,Blood & Water,[],"[InternationalTVShows, TVDramas, TVMysteries]","[After, crossing, paths, at, a, party,, a, Cap..."
2,s3,Ganglands,[JulienLeclercq],"[CrimeTVShows, InternationalTVShows, TVAction&...","[To, protect, his, family, from, a, powerful, ..."
3,s4,Jailbirds New Orleans,[],"[Docuseries, RealityTV]","[Feuds,, flirtations, and, toilet, talk, go, d..."
4,s5,Kota Factory,[],"[InternationalTVShows, RomanticTVShows, TVCome...","[In, a, city, of, coaching, centers, known, to..."


### Preparando os Dados Para Vetorização em Outro Espaço Vetorial

Para a vetorização, faz-se necessária a unificação dos textos descritivos em uma única variável, que aqui será definida como "tags"

In [ ]:
df['tags'] = df['director'] + \
                        df['listed_in'] + \
                        df['description']

In [27]:
df_final = df[["show_id", "title", "tags"]]

In [29]:
df_final.head()

,show_id,title,tags
0,s1,Dick Johnson Is Dead,"[KirstenJohnson, Documentaries, As, her, fathe..."
1,s2,Blood & Water,"[, InternationalTVShows, TVDramas, TVMysteries..."
2,s3,Ganglands,"[JulienLeclercq, CrimeTVShows, InternationalTV..."
3,s4,Jailbirds New Orleans,"[, Docuseries, RealityTV, Feuds,, flirtations,..."
4,s5,Kota Factory,"[, InternationalTVShows, RomanticTVShows, TVCo..."


Para finalizar, será feita uma junção dos elementos das listas em "tags", com o objetivo de simplificar posteriormente a vetorização. Após isso, o texto será transformado para conter apenas letras minúsculas.

In [30]:
df_final['tags'] = df_final['tags'].apply(lambda x:" ".join(x))
df_final['tags'] = df_final['tags'].apply(lambda x:x.lower())

In [31]:
df_final.head()

,show_id,title,tags
0,s1,Dick Johnson Is Dead,kirstenjohnson documentaries as her father nea...
1,s2,Blood & Water,internationaltvshows tvdramas tvmysteries aft...
2,s3,Ganglands,julienleclercq crimetvshows internationaltvsho...
3,s4,Jailbirds New Orleans,"docuseries realitytv feuds, flirtations and t..."
4,s5,Kota Factory,internationaltvshows romantictvshows tvcomedi...


In [32]:
df_final.to_csv("../data/processed/netflix_titles_processed.csv", index=False)

A modelagem final foi modularizada em scripts dentro de src/.